# Assignment 3: comparative machine translation

Run all cells after selecting a GPU runtime. The workflow is resumable and validates saved artifacts; a zero exit code alone is not treated as success. It uses the fixed dataset split and runs the repository pipeline without duplicating model code.

In [ ]:
# Initial setup: clone/update code, install dependencies, and select the persistent Drive run directory.
REPO_URL = 'https://github.com/sruthiidv/machine-translation-model-comparison.git'
REPO_REF = 'main'
from pathlib import Path
import os, subprocess
repo = Path('/content/machine-translation-model-comparison')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = Path('/content/drive/MyDrive/mt_translation_assignment/results')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('repo:', repo)
print('run directory:', RUN_DIR)

In [ ]:
# Required authorizations and resource checks. Nothing secret is printed or written to the repository.
import torch
assert torch.cuda.is_available(), 'Enable Runtime > Change runtime type > GPU before Run all.'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), 'GiB')
from google.colab import userdata
token = userdata.get('HF_TOKEN')
assert token, 'Add HF_TOKEN to Colab Secrets after accepting the LLaMA and Gemma licenses.'
os.environ['HF_TOKEN'] = token
import shutil
free_gib = shutil.disk_usage(RUN_DIR).free / 2**30
print('free Drive space:', round(free_gib, 2), 'GiB')
assert free_gib >= 4, 'Free at least 4 GiB on Drive before starting the model-only mT5 run.'

In [ ]:
# Optional cleanup: deliberately targets only known failed mT5 checkpoint directories.
# It never touches the Transformer, data, reports, or unrelated Drive files.
CLEAN_FAILED_MT5 = False
if CLEAN_FAILED_MT5:
    import shutil
    for name in ('checkpoint-313', 'checkpoint-626', 'checkpoint-939'):
        target = RUN_DIR / 'models' / 'mt5' / name
        if target.exists() and target.is_dir():
            print('removing explicitly named failed checkpoint:', target)
            shutil.rmtree(target)
else:
    print('Cleanup disabled; failed-run logs remain preserved.')

In [ ]:
# One resumable Run-all operation. It restores compatible artifacts, saves predictions incrementally,
# loads large models sequentially, and fails loudly if required evidence is missing or non-finite.
import subprocess
command = ['python', '-m', 'scripts.run_pipeline', '--stage', 'all', '--output-dir', str(RUN_DIR), '--reports-dir', str(RUN_DIR / 'reports')]
completed = subprocess.run(command, text=True)
if completed.returncode != 0:
    raise RuntimeError('Pipeline failed. Read the explicit error above; rerun this cell after fixing the reported condition.')
print('Pipeline and artifact validation completed.')

In [ ]:
# Show the evidence generated from real outputs and create a Drive archive without model weights.
import pandas as pd, zipfile
metrics = pd.read_csv(RUN_DIR / 'metrics.csv')
display(metrics)
required = [RUN_DIR / 'metrics.csv', RUN_DIR / 'predictions' / 'all_predictions.jsonl', RUN_DIR / 'ethical_review.jsonl', RUN_DIR / 'reports' / 'model_comparison.md']
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing final artifacts: {missing}'
zip_path = RUN_DIR.parent / 'mt_translation_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_DIR.rglob('*'):
        if path.is_file() and not any(part in {'models', 'failed_runs'} for part in path.parts) and path.suffix not in {'.pt', '.bin', '.safetensors'}:
            archive.write(path, path.relative_to(RUN_DIR))
print('submission archive:', zip_path)
print((RUN_DIR / 'reports' / 'ethical_analysis.md').read_text(encoding='utf-8')[:3000])